from notebooks.unit2.unit2_m5 import gammafrom notebooks.unit2.unit2_m5 import learning_ratefrom random import random# 本单元的本地版本（Apple Silicon / M 系列芯片）

本笔记本是 [unit2.ipynb](unit2.ipynb) 的 **macOS 本地版**，与 Colab 版的差异：

- 删掉 `apt-get` / `xvfb` / `pyvirtualdisplay` 虚拟屏 —— 这些是 Linux 专属的；FrozenLake 和 Taxi 的 `rgb_array` 渲染只用 pygame 的离屏 Surface，Mac 上不需要虚拟屏。
- 删掉强制重启内核的 `os.kill(os.getpid(), 9)` —— 在本地会直接杀掉 Jupyter 内核。
- `pickle5` → 标准库 `pickle`（`pickle5` 在 Python 3.12 上装不上，也已无必要）。
- 去掉 Hugging Face Hub 上传流程：模型存本地 `.pkl`，回放存本地 `.mp4` 并在笔记本里内联播放。

准备环境（项目根目录）：

```bash
uv sync
```

然后在 Jupyter 中选择项目 `.venv` 作为内核。本单元是纯 NumPy 的 Q 学习，不需要 GPU/MPS。

# 第二单元：使用FrozenLake-v1 ⛄ 和 Taxi-v3 🚕 进行Q学习

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/thumbnail.jpg" alt="Unit 2 Thumbnail">

在这个笔记本中，**你将从零开始编写你的第一个强化学习智能体**，使用Q学习玩FrozenLake ❄️，并与社区分享，以及尝试不同的配置。

⬇️ 这是一个你**在几分钟内就能实现**的例子。⬇️

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/envs.gif" alt="环境"/>

###🎮 环境：

- [FrozenLake-v1](https://gymnasium.farama.org/environments/toy_text/frozen_lake/)
- [Taxi-v3](https://gymnasium.farama.org/environments/toy_text/taxi/)

###📚 RL-库：

- Python 和 NumPy
- [Gymnasium](https://gymnasium.farama.org/)

我们一直在努力改进教程，所以**如果你发现本笔记本中的一些问题**，请[在GitHub仓库上提出问题](https://github.com/huggingface/deep-rl-class/issues)。

## 本笔记本的目标 🏆

完成本笔记本后，你将能够：

- 使用**Gymnasium**，环境库。
- 从头开始编写一个Q学习智能体。
- **将你训练好的智能体和代码推送到Hub**，附带精美的视频回放和评估分数 🔥。

在这个免费课程中，你将：

- 📖 **理论和实践**学习深度强化学习。
- 🧑‍💻 学习使用**著名的深度强化学习库**，如Stable Baselines3、RL Baselines3 Zoo、CleanRL和Sample Factory 2.0。
- 🤖 在**独特环境中训练智能体**

更多信息请查看📚 课程大纲 👉 https://simoninithomas.github.io/deep-rl-course

别忘了**<a href="http://eepurl.com/ic5ZUD">注册课程</a>**（我们正在收集你的电子邮件，以便**在每个单元发布时向你发送链接，并向你提供有关挑战和更新的信息）。**

保持联系的最佳方式是加入我们的Discord服务器，与社区和我们交流 👉🏻 https://discord.gg/ydHrjt3WP5

## 先决条件 🏗️

在深入学习笔记本之前，你需要：

🔲 📚 **阅读单元2，学习Q学习** [Q-Learning by reading Unit 2](https://huggingface.co/deep-rl-course/unit2/introduction) 🤗

## Q学习小结

*Q学习* **是一种强化学习算法，它**：

- 训练*Q函数*，这是一种**动作值函数**，通过*Q表*在内部内存中编码，该**Q表包含所有状态-动作对的值。**
    
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-function-2.jpg" alt="Q function"  width="100%"/>

- 给定一个状态和动作，我们的Q函数**将在Q表中搜索相应的值。**
    
- 当训练完成时，**我们拥有一个最优的Q函数，因此也拥有一个最优的Q表。**
    
- 如果我们**有一个最优的Q函数**，我们就有了一个最优的策略，因为我们**知道对于每个状态，应该采取的最佳动作。**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/link-value-policy.jpg" alt="Link value policy"  width="100%"/>


但是，在一开始，我们的**Q表是无用的，因为它为每个状态-动作对赋予了任意值（大多数情况下我们将Q表初始化为0）**。但是，随着我们探索环境并更新Q表，它将为我们提供越来越好的近似值。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/q-learning.jpeg" alt="q-learning.jpeg" width="100%"/>

这是Q学习的伪代码：

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q-Learning" width="100%"/>

# 让我们编写我们的第一个强化学习算法 🚀

要验证此动手练习是否符合[认证流程](https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process)的要求，你需要将训练好的Taxi模型推送到Hub并**获得 >= 4.5 的结果**。

要查找你的结果，请访问[排行榜](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard)并找到你的模型，**结果 = 平均奖励 - 奖励标准差**

有关认证流程的更多信息，请查看此部分 👉 https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process

## 依赖 🔽

推荐用项目根目录的 `uv sync` 安装。如果你想直接装进当前内核，可以运行下面这个单元格。

我们会用到：

- `gymnasium`：包含 FrozenLake-v1 ⛄ 和 Taxi-v3 🚕 环境。
- `pygame`：FrozenLake-v1 和 Taxi-v3 的渲染。
- `numpy`：处理我们的 Q 表。
- `imageio` / `imageio_ffmpeg`：把回放帧写成 mp4。
- `tqdm`、`ipywidgets`：训练进度条。
- `huggingface_hub`：第三部分从 Hub 下载别人训练好的模型（只下载，不需要登录）。

In [ ]:
%pip install gymnasium pygame numpy imageio imageio_ffmpeg tqdm ipywidgets huggingface_hub

## 导入包 📦

除了已安装的库，我们还使用：

- `random`：用于生成随机数（对epsilon-greedy策略很有用）。
- `imageio`：用于生成重播视频。

In [2]:
import os
import random
import pickle

import numpy as np
import gymnasium as gym
import imageio
from tqdm.auto import tqdm
from IPython.display import Video

# 如果 pygame 在无窗口环境下报 SDL 错误，取消下面这行的注释
# os.environ["SDL_VIDEODRIVER"] = "dummy"

我们现在准备编写Q学习算法 🔥

# 第一部分：冰湖 ⛄ (非湿滑版本)

## 创建并理解[冰湖环境 ⛄]((https://gymnasium.farama.org/environments/toy_text/frozen_lake/)
---

💡 开始使用环境时，一个好习惯是查看其文档

👉 https://gymnasium.farama.org/environments/toy_text/frozen_lake/

---

我们将训练我们的Q学习智能体**通过只在冰冻的瓷砖（F）上行走并避开洞（H）来从起始状态（S）导航到目标状态（G）**。

我们可以有两种大小的环境：

- `map_name="4x4"`：一个4x4的网格版本
- `map_name="8x8"`：一个8x8的网格版本


环境有两种模式：

- `is_slippery=False`：由于冰湖的非湿滑性质（确定性），智能体始终**按预期方向移动**。
- `is_slippery=True`：由于冰湖的湿滑性质（随机性），智能体**可能不总是按预期方向移动**。

现在让我们保持简单，使用4x4地图和非湿滑版本。
我们添加了一个名为`render_mode`的参数，它指定了环境应如何可视化。在我们的例子中，因为我们**希望在最后录制环境的视频，所以我们需要将`render_mode`设置为`rgb_array`**。

正如[文档中所解释的](https://gymnasium.farama.org/api/env/#gymnasium.Env.render)，“rgb_array”：返回一个代表环境当前状态的单帧。帧是一个`np.ndarray`，其形状为(x, y, 3)，表示x乘y像素图像的RGB值。

**练习**（下面是待填空的代码，答案见下一节「解决方案」）：

```python
# Create the FrozenLake-v1 environment using 4x4 map and non-slippery version and render_mode="rgb_array"
env = gym.make() # TODO use the correct parameters
```

### 解决方案

In [ ]:
desc=["SFFF",
      "FHFH",
      "FFFH",
      "HFFG"]
env = gym.make("FrozenLake-v1", desc=desc, is_slippery=True, render_mode="rgb_array")

你可以像这样创建自己的自定义网格：

```python
desc=["SFFF", "FHFH", "FFFH", "HFFG"]
gym.make('FrozenLake-v1', desc=desc, is_slippery=True)
```

但我们现在将使用默认环境。

### 让我们看看环境是什么样子的：

In [ ]:
# We create our environment with gym.make("<name_of_the_environment>")- `is_slippery=False`: The agent always moves in the intended direction due to the non-slippery nature of the frozen lake (deterministic).
print("_____OBSERVATION SPACE_____ \n")
print("Observation Space", env.observation_space)
print("Sample observation", env.observation_space.sample()) # Get a random observation

我们看到`Observation Space Shape Discrete(16)`表示观察是一个整数，代表**智能体的当前位置为`current_row * ncols + current_col`（其中行和列都从0开始）**。

例如，在4x4地图中，目标位置可以按如下方式计算：3 * 4 + 3 = 15。可能的观察数量取决于地图的大小。**例如，4x4地图有16种可能的观察。**


例如，state = 0 看起来像这样：

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/frozenlake.png" alt="FrozenLake">

In [ ]:
print("\n _____ACTION SPACE_____ \n")
print("Action Space Shape", env.action_space.n)
print("Action Space Sample", env.action_space.sample()) # Take a random action

动作空间（智能体可以采取的可能动作集）是离散的，有4个可用动作🎮：
- 0：向左移动
- 1：向下移动
- 2：向右移动
- 3：向上移动

奖励函数💰：
- 到达目标：+1
- 掉入洞中：0
- 到达冰面：0

## 创建并初始化Q表 🗄️

(👀 伪代码的步骤1)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q-Learning" width="100%"/>


是时候初始化我们的Q表了！为了知道要使用多少行（状态）和列（动作），我们需要知道动作空间和观察空间。我们之前已经知道它们的值，但我们希望以编程方式获取它们，以便我们的算法适用于不同的环境。Gym提供了这样做的方法：`env.action_space.n`和`env.observation_space.n`

**练习**（下面是待填空的代码，答案见下一节「解决方案」）：

```python
state_space =
print("There are ", state_space, " possible states")

action_space =
print("There are ", action_space, " possible actions")
```

**练习**（下面是待填空的代码，答案见下一节「解决方案」）：

```python
# Let's create our Qtable of size (state_space, action_space) and initialized each values at 0 using np.zeros. np.zeros needs a tuple (a,b)
def initialize_q_table(state_space, action_space):
  Qtable =
  return Qtable
```

**练习**（下面是待填空的代码，答案见下一节「解决方案」）：

```python
Qtable_frozenlake = initialize_q_table(state_space, action_space)
```

### 解决方案

In [ ]:
state_space = env.observation_space.n
print("There are ", state_space, " possible states")

action_space = env.action_space.n
print("There are ", action_space, " possible actions")

In [6]:
# Let's create our Qtable of size (state_space, action_space) and initialized each values at 0 using np.zeros
def initialize_q_table(state_space, action_space):
  Qtable = np.zeros((state_space, action_space))
  return Qtable

In [ ]:
Qtable_frozenlake = initialize_q_table(state_space, action_space)

## 定义贪婪策略 🤖

请记住，我们有两种策略，因为Q学习是一种**离策略**算法。这意味着我们正在使用**不同的策略来行动和更新值函数**。

- Epsilon-greedy策略（行动策略）
- 贪婪策略（更新策略）

贪婪策略也将是Q学习智能体完成训练时我们拥有的最终策略。贪婪策略用于使用Q表选择动作。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/off-on-4.jpg" alt="Q-Learning" width="100%"/>

**练习**（下面是待填空的代码，答案见下一节「解决方案」）：

```python
def greedy_policy(Qtable, state):
  # Exploitation: take the action with the highest state, action value
  action = np.argmax(Qtable[state][:])

  return action
```

#### 解决方案

In [14]:
def greedy_policy(Qtable, state):
  # Exploitation: take the action with the highest state, action value
  action = np.argmax(Qtable[state][:])

  return action

## 定义epsilon-greedy策略 🤖

Epsilon-greedy是处理探索/利用权衡的训练策略。

epsilon-greedy的想法是：

- 以*概率1 - ɛ*：**我们进行利用**（即我们的智能体选择具有最高状态-动作对值的动作）。

- 以*概率ɛ*：我们进行**探索**（尝试一个随机动作）。

随着训练的继续，我们逐渐**减少epsilon值，因为我们需要越来越少的探索和越来越多的利用。**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-4.jpg" alt="Q-Learning" width="100%"/>

**练习**（下面是待填空的代码，答案见下一节「解决方案」）：

```python
import random
def epsilon_greedy_policy(Qtable, state, epsilon):
  # Randomly generate a number between 0 and 1
  random_num = random.uniform(0, 1)
  # if random_num > greater than epsilon --> exploitation
  if random_num > epsilon:
    # Take the action with the highest value given a state
    # np.argmax can be useful here
    action = np.argmax(Qtable[state][:])
  # else --> exploration
  else:
    action = env.action_space.sample() # Take a random action

  return action
```

#### 解决方案

In [12]:
def epsilon_greedy_policy(Qtable, state, epsilon):
  # Randomly generate a number between 0 and 1
  random_num = random.uniform(0,1)
  # if random_num > greater than epsilon --> exploitation
  if random_num > epsilon:
    # Take the action with the highest value given a state
    # np.argmax can be useful here
    action = greedy_policy(Qtable, state)
  # else --> exploration
  else:
    action = env.action_space.sample()

  return action

## 定义超参数 ⚙️

与探索相关的超参数是一些最重要的参数。

- 我们需要确保我们的智能体**充分探索状态空间**以学习一个好的值近似。为此，我们需要逐步衰减epsilon。
- 如果你衰减epsilon过快（衰减率过高），**你将面临智能体卡住的风险**，因为你的智能体没有充分探索状态空间，因此无法解决问题。

In [ ]:
# Training parameters
n_training_episodes = 10000  # Total training episodes
learning_rate = 0.7          # Learning rate

# Evaluation parameters
n_eval_episodes = 100        # Total number of test episodes

# Environment parameters
env_id = "FrozenLake-v1"     # Name of the environment
max_steps = 99               # Max steps per episode
gamma = 0.95                 # Discounting rate
eval_seed = []               # The evaluation seed of the environment

# Exploration parameters
max_epsilon = 1.0             # Exploration probability at start
min_epsilon = 0.05            # Minimum exploration probability
decay_rate = 0.0005            # Exponential decay rate for exploration prob

## 创建训练循环方法

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit3/Q-learning-2.jpg" alt="Q-Learning" width="100%"/>

训练循环如下：

```
对于总训练集中的每个episode：

  减少epsilon（因为我们需要越来越少的探索）
  重置环境

  对于最大时间步中的每一步：
    使用epsilon-greedy策略选择动作At
    执行动作(a)并观察结果状态(s')和奖励(r)
    使用Bellman方程更新Q值Q(s,a) = Q(s,a) + lr * [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
    如果完成，结束episode
    我们的下一个状态是新状态
```

**练习**（下面是待填空的代码，答案见下一节「解决方案」）：

```python
import numpy as np
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
  for episode in tqdm(range(n_training_episodes)):
    # Reduce epsilon (because we need less and less exploration)
    epsilon = min_epsilon + (max_epsilon - min_epsilon)*np.exp(-decay_rate*episode)
    # Reset the environment
    state, info = env.reset()
    step = 0
    terminated = False
    truncated = False

    # repeat
    for step in range(max_steps):
      # Choose the action At using epsilon greedy policy
      action = epsilon_greedy_policy(Qtable, state)

      # Take action At and observe Rt+1 and St+1
      # Take the action (a) and observe the outcome state(s') and reward (r)
      new_state, reward, terminated, truncated, info = env.step(action)

      # Update Q(s,a):= Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
      Qtable[state][action] = Qtable[state][action] + learning_rate * (reward + gamma * np.max(Qtable[new_state]) - Qtable[state][action])

      # If terminated or truncated finish the episode
      if terminated or truncated:
        break

      # Our next state is the new state
      state = new_state
  return Qtable
```

#### 解决方案

In [10]:
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
  for episode in tqdm(range(n_training_episodes)):
    # Reduce epsilon (because we need less and less exploration)
    epsilon = min_epsilon + (max_epsilon - min_epsilon)*np.exp(-decay_rate*episode)
    # Reset the environment
    state, info = env.reset()
    step = 0
    terminated = False
    truncated = False

    # repeat
    for step in range(max_steps):
      # Choose the action At using epsilon greedy policy
      action = epsilon_greedy_policy(Qtable, state, epsilon)

      # Take action At and observe Rt+1 and St+1
      # Take the action (a) and observe the outcome state(s') and reward (r)
      new_state, reward, terminated, truncated, info = env.step(action)

      # Update Q(s,a):= Q(s,a) + lr [R(s,a) + gamma * max Q(s',a') - Q(s,a)]
      Qtable[state][action] = Qtable[state][action] + learning_rate * (reward + gamma * np.max(Qtable[new_state]) - Qtable[state][action])

      # If terminated or truncated finish the episode
      if terminated or truncated:
        break

      # Our next state is the new state
      state = new_state
  return Qtable

## 训练Q学习智能体 🏃

In [ ]:
Qtable_frozenlake = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_frozenlake)

## 让我们看看现在的Q学习表是什么样子 👀

In [ ]:
Qtable_frozenlake

## 评估方法 📝

- 我们定义了将用于测试Q学习智能体的评估方法。

In [17]:
def evaluate_agent(env, max_steps, n_eval_episodes, Q, seed):
  """
  Evaluate the agent for ``n_eval_episodes`` episodes and returns average reward and std of reward.
  :param env: The evaluation environment
  :param max_steps: Maximum number of steps per episode
  :param n_eval_episodes: Number of episode to evaluate the agent
  :param Q: The Q-table
  :param seed: The evaluation seed array (for taxi-v3)
  """
  episode_rewards = []
  for episode in tqdm(range(n_eval_episodes)):
    if seed:
      state, info = env.reset(seed=seed[episode])
    else:
      state, info = env.reset()
    step = 0
    truncated = False
    terminated = False
    total_rewards_ep = 0

    for step in range(max_steps):
      # Take the action (index) that have the maximum expected future reward given that state
      action = greedy_policy(Q, state)
      new_state, reward, terminated, truncated, info = env.step(action)
      total_rewards_ep += reward

      if terminated or truncated:
        break
      state = new_state
    episode_rewards.append(total_rewards_ep)
  mean_reward = np.mean(episode_rewards)
  std_reward = np.std(episode_rewards)

  return mean_reward, std_reward

## 评估我们的Q学习智能体 📈

- 通常，你的平均奖励应为1.0。
- **环境相对简单**，因为状态空间非常小（16）。你可以尝试[将其替换为湿滑版本](https://gymnasium.farama.org/environments/toy_text/frozen_lake/)，这将引入随机性，使环境更加复杂。

In [ ]:
# Evaluate our Agent
mean_reward, std_reward = evaluate_agent(env, max_steps, n_eval_episodes, Qtable_frozenlake, eval_seed)
print(f"Mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

## 保存模型 💾 并录制回放 🎥

- 模型（超参数 + Q 表）用标准库 `pickle` 存成本地文件，方便复现。
- `record_video` 用 `render_mode="rgb_array"` 逐帧渲染并写成 mp4。

> 注：原版 `record_video` 的循环条件 `while not terminated or truncated` 有 bug（`truncated=True` 时会死循环），这里改成 `while not (terminated or truncated)` 并加了步数上限。

In [20]:
def record_video(env, Qtable, out_directory, fps=1, max_steps=200):
  """
  生成智能体的回放视频
  :param env: 环境（render_mode 必须是 "rgb_array"）
  :param Qtable: 智能体的 Q 表
  :param out_directory: 视频输出路径
  :param fps: 每秒帧数（taxi-v3 和 frozenlake-v1 用 1）
  :param max_steps: 单个 episode 的最大步数，防止策略绕圈时无限录制
  """
  images = []
  terminated = False
  truncated = False
  state, info = env.reset(seed=random.randint(0, 500))
  img = env.render()
  images.append(img)
  step = 0
  while not (terminated or truncated) and step < max_steps:
    # 选取在该状态下期望未来奖励最大的动作
    action = np.argmax(Qtable[state][:])
    state, reward, terminated, truncated, info = env.step(action)
    img = env.render()
    images.append(img)
    step += 1
  imageio.mimsave(out_directory, [np.array(img) for i, img in enumerate(images)], fps=fps)

In [ ]:
model = {
    "env_id": env_id,
    "max_steps": max_steps,
    "n_training_episodes": n_training_episodes,
    "n_eval_episodes": n_eval_episodes,
    "eval_seed": eval_seed,

    "learning_rate": learning_rate,
    "gamma": gamma,

    "max_epsilon": max_epsilon,
    "min_epsilon": min_epsilon,
    "decay_rate": decay_rate,

    "qtable": Qtable_frozenlake
}

with open("q-FrozenLake-v1-4x4-noSlippery.pkl", "wb") as f:
    pickle.dump(model, f)

record_video(env, model["qtable"], "replay-frozenlake.mp4", fps=1)
Video("replay-frozenlake.mp4", embed=True)

恭喜🥳 你刚刚从头开始实现、训练并上传了你的第一个强化学习智能体。
FrozenLake-v1 non-slippery是一个非常简单的环境，让我们尝试一个更难的环境 🔥。

# 第二部分：出租车-v3 🚖

## 创建并理解[出租车-v3 🚕](https://gymnasium.farama.org/environments/toy_text/taxi/)
---

💡 开始使用环境时，一个好习惯是查看其文档

👉 https://gymnasium.farama.org/environments/toy_text/taxi/

---

在`Taxi-v3` 🚕中，网格世界中有四个指定位置，分别用R（红色）、G（绿色）、Y（黄色）和B（蓝色）表示。

当episode开始时，**出租车从一个随机方块开始**，乘客在一个随机位置。出租车开到乘客的位置，**接载乘客**，开到乘客的目的地（四个指定位置中的另一个），然后**放下乘客**。一旦乘客放下，episode结束。


<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/taxi.png" alt="Taxi">

> **版本提示**：从 gymnasium 1.3.0 起 `Taxi-v3` 已被 `Taxi-v4` 取代，直接 `gym.make("Taxi-v3")` 会抛 `DeprecatedEnv`。
>
> `Taxi-v4` 的默认参数（`is_rainy=False`、`fickle_passenger=False`）与 v3 行为一致，所以下面全部改用 `Taxi-v4`。

In [3]:
env = gym.make("Taxi-v4", render_mode="rgb_array")

由于有25个出租车位置、5个可能的乘客位置（包括乘客在出租车内的情况）和4个目的地位置，因此有**500个离散状态**。

In [4]:
state_space = env.observation_space.n
print("There are ", state_space, " possible states")

There are  500  possible states


In [5]:
action_space = env.action_space.n
print("There are ", action_space, " possible actions")

There are  6  possible actions


动作空间（智能体可以采取的可能动作集）是离散的，有**6个可用动作🎮**：

- 0：向南移动
- 1：向北移动
- 2：向东移动
- 3：向西移动
- 4：接载乘客
- 5：放下乘客

奖励函数💰：

- 每步-1，除非触发其他奖励。
- 成功送达乘客+20。
- 非法执行“接载”和“放下”动作-10。

In [7]:
# Create our Q table with state_size rows and action_size columns (500x6)
Qtable_taxi = initialize_q_table(state_space, action_space)
print(Qtable_taxi)
print("Q-table shape: ", Qtable_taxi .shape)

[[0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 ...
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]
Q-table shape:  (500, 6)


## 定义超参数 ⚙️

⚠ 不要修改 EVAL_SEED：`eval_seed`数组**允许我们用相同的出租车起始位置评估你的智能体，以确保所有同学都获得相同的评估结果**

In [8]:
# Training parameters
n_training_episodes = 25000   # Total training episodes
learning_rate = 0.7           # Learning rate

# Evaluation parameters
n_eval_episodes = 100        # Total number of test episodes

# DO NOT MODIFY EVAL_SEED
eval_seed = [16,54,165,177,191,191,120,80,149,178,48,38,6,125,174,73,50,172,100,148,146,6,25,40,68,148,49,167,9,97,164,176,61,7,54,55,
 161,131,184,51,170,12,120,113,95,126,51,98,36,135,54,82,45,95,89,59,95,124,9,113,58,85,51,134,121,169,105,21,30,11,50,65,12,43,82,145,152,97,106,55,31,85,38,
 112,102,168,123,97,21,83,158,26,80,63,5,81,32,11,28,148] # Evaluation seed, this ensures that all classmates agents are trained on the same taxi starting position
                                                          # Each seed has a specific starting state

# Environment parameters
env_id = "Taxi-v4"           # Name of the environment
max_steps = 99               # Max steps per episode
gamma = 0.95                 # Discounting rate

# Exploration parameters
max_epsilon = 1.0             # Exploration probability at start
min_epsilon = 0.05           # Minimum exploration probability
decay_rate = 0.005            # Exponential decay rate for exploration prob


## 训练我们的Q学习智能体 🏃

In [15]:
Qtable_taxi = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable_taxi)
Qtable_taxi

  0%|          | 0/25000 [00:00<?, ?it/s]

array([[  0.        ,   0.        ,   0.        ,   0.        ,
          0.        ,   0.        ],
       [  2.75200369,   3.94947757,   2.75200369,   3.94947754,
          5.20997639,  -5.05052243],
       [  7.93349184,   9.40367562,   7.93349184,   9.40367562,
         10.9512375 ,   0.40367554],
       ...,
       [ -1.7178    ,  -1.59705   ,  -2.106391  ,   9.40367562,
         -7.18442162,  -9.5655    ],
       [ -5.32807644,   3.08416623,   2.03020263,   6.53681725,
        -12.71610719, -13.23583975],
       [ 10.99699999,  13.95111571,  13.71494575,  18.        ,
          2.051     ,   7.05267102]], shape=(500, 6))

## 评估我们的出租车智能体 📈

用固定的 `eval_seed` 评估，**结果 = 平均奖励 - 奖励标准差**，课程认证要求 >= 4.5。

In [18]:
mean_reward, std_reward = evaluate_agent(env, max_steps, n_eval_episodes, Qtable_taxi, eval_seed)
print(f"Mean_reward={mean_reward:.2f} +/- {std_reward:.2f}")

  0%|          | 0/100 [00:00<?, ?it/s]

Mean_reward=7.56 +/- 2.71


In [21]:
model = {
    "env_id": env_id,
    "max_steps": max_steps,
    "n_training_episodes": n_training_episodes,
    "n_eval_episodes": n_eval_episodes,
    "eval_seed": eval_seed,

    "learning_rate": learning_rate,
    "gamma": gamma,

    "max_epsilon": max_epsilon,
    "min_epsilon": min_epsilon,
    "decay_rate": decay_rate,

    "qtable": Qtable_taxi
}

with open("q-Taxi-v4.pkl", "wb") as f:
    pickle.dump(model, f)

record_video(env, model["qtable"], "replay-taxi.mp4", fps=1)
Video("replay-taxi.mp4", embed=True)

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (550, 350) to (560, 352) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


# 第三部分：从 Hub 加载 🔽

Hugging Face Hub 🤗 的惊人之处在于你可以轻松地从社区加载强大的模型。**下载不需要登录**。

1. 访问 https://huggingface.co/models?other=q-learning 查看所有 Q 学习已保存模型的列表。
2. 选择一个并复制其 repo_id

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit2/copy-id.png" alt="Copy id">

3. 然后我们只需使用`load_from_hub`，并提供：
- repo_id
- 文件名：仓库中保存的模型文件。

#### 不要修改此代码

In [ ]:
from urllib.error import HTTPError

from huggingface_hub import hf_hub_download


def load_from_hub(repo_id: str, filename: str) -> str:
    """
    Download a model from Hugging Face Hub.
    :param repo_id: id of the model repository from the Hugging Face Hub
    :param filename: name of the model zip file from the repository
    """
    # Get the model from the Hub, download and cache the model on your local disk
    pickle_model = hf_hub_download(
        repo_id=repo_id,
        filename=filename
    )

    with open(pickle_model, 'rb') as f:
      downloaded_model_file = pickle.load(f)

    return downloaded_model_file

### .

In [ ]:
model = load_from_hub(repo_id="ThomasSimonini/q-Taxi-v3", filename="q-learning.pkl") # Try to use another model

print(model)
env_id_compat = "Taxi-v4" if model["env_id"] == "Taxi-v3" else model["env_id"]  # 老模型存的是 v3
env = gym.make(env_id_compat)

evaluate_agent(env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"])

In [ ]:
model = load_from_hub(repo_id="ThomasSimonini/q-FrozenLake-v1-no-slippery", filename="q-learning.pkl") # Try to use another model

env = gym.make(model["env_id"], is_slippery=False)

evaluate_agent(env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"])

## 一些额外的挑战 🏆

学习**最好的方式就是亲自尝试**！第一个建议是训练更多步，在 1,000,000 步之后能看到很好的结果。

- 训练更多步
- 尝试不同的超参数
- 换个环境玩玩，比如 FrozenLake-v1 的湿滑（`is_slippery=True`）版本，[查阅 gymnasium 文档](https://gymnasium.farama.org/)

如果你想把模型推送到 [排行榜](https://huggingface.co/spaces/huggingface-projects/Deep-Reinforcement-Learning-Leaderboard)，用 Colab 版 [unit2.ipynb](unit2.ipynb) 里的 `push_to_hub` 流程即可（本地版已去掉上传部分）。

_____________________________________________________________________
恭喜🥳，你刚刚从头实现、训练并上传了你的第一个强化学习智能体。

理解Q学习是**理解基于值的方法的重要一步。**

在下一个单元，深度Q学习中，我们将看到虽然创建和更新Q表是一个很好的策略——**然而，它不可扩展。**

例如，想象你创建一个智能体来玩Doom游戏。

<img src="https://vizdoom.cs.put.edu.pl/user/pages/01.tutorial/basic.png" alt="Doom"/>

Doom是一个拥有巨大状态空间（数百万个不同状态）的大型环境。为该环境创建和更新Q表将是低效的。

这就是为什么我们将在下一个单元学习深度Q学习，这是一种**我们使用神经网络来近似给定状态下每个动作的Q值**的算法。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit4/atari-envs.gif" alt="Environments"/>

单元3见！🔥

## 继续学习，保持出色 🤗